# Cohort Creation Workflow (Steps 1b → 2)

Final workflow for creating cohorts. One cell per cohort series; run Configuration once, then the cohort cell(s) you need.

## Order of operations

1. **Configuration** — Set project root and paths (run once).
2. **Cohort 1 (FALLS)** — Builds all age-band × event-year partitions for the `fall_injury_any` outcome.
3. **Cohort 2 (ED)** — Builds all age-band × event-year partitions for the `ed_event` outcome.
4. **Check status** — Pipeline state and durations in S3; cohort parquets in S3 data lake. Use `--outputs` for file listing; use `--logs` for build logs.

## Cohorts

| Series | Script | Target |
|--------|--------|--------|
| **FALLS** | `run_series_falls.py` | `fall_injury_any` (injury S00–S99 + external cause W00–W19 for the same patient within `CPIC_FALL_TARGET_WINDOW_DAYS`, default 7 days) |
| **ED** | `run_series_ed.py` | `ed_event` (POS=23 or revenue code 045x/0981) |

## Prerequisites

- **Step 1a** — APCD input data in gold (medical/pharmacy).
- **Step 1b** — Event filter producing `fall_injury_any`, `fall_injury_serious`, `fall_injury_head`, `ed_event` flags plus feature flags (`r29_6_flag`, `z91_81_flag`, `cpt_1100f_flag`).

## Configuration

In [ ]:
import sys
from pathlib import Path

# Project root (notebook at repo root)
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PYTHON_BIN = Path(sys.executable)
LOG_DIR = PROJECT_ROOT / "2_create_cohort" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {PYTHON_BIN}")
print(f"Log dir: {LOG_DIR}")

## Cohort 1: FALLS

Runs the FALLS series for all age bands and event years. Each partition writes `fall_injury_any` cohort parquets to S3. Partitions that already have the cohort output are skipped (`--skip-existing`).

In [ ]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_falls.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

## Cohort 2: ED

Runs the ED series for all age bands and event years. Partitions that already have the cohort output are skipped (`--skip-existing`). Use this cell to run ED-only or to force-rebuild (omit `--skip-existing` if needed).

In [ ]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_ed.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)

## Check S3 completion status

Lists pipeline state (pgx-repository) with status and **duration** per partition, and cohort parquet counts in pgxdatalake. Add `--outputs` to the script args to list each parquet with LastModified and size; add `--logs` to list build logs from pgx-repository.

In [ ]:
import subprocess
script = PROJECT_ROOT / "2_create_cohort" / "check_s3_cohort_completion.py"
# Use --outputs to list each cohort.parquet with LastModified and size
subprocess.run([str(PYTHON_BIN), str(script), "--outputs"], cwd=str(PROJECT_ROOT))